# Introduction to Profiling — Intel VTune
## Concepts and practice · NCI ARE · 12 cores on one node

**This notebook runs inside the JupyterLab session you submit through ARE**, NCI's customised Open OnDemand service. Select **12 CPU cores on one node** in the ARE launch form. The running session supplies the resources for timing and profiling the **prebuilt LULESH application provided through the private `lulesh` module for project `vp91`**.

Follow the MPI workshop format: **read a concept, predict, run an exercise, check the evidence, then continue**. The application is LULESH, as in the [setup RST](../docs/source/tutorial/intro.rst) and [VTune RST](../docs/source/tutorial/vtune.rst). The [slides](../docs/slides/NCI_VTune_2026.pdf) accompany the profiling concepts; use this notebook's **12-core ARE configuration** for the practical exercises.

**By the end you should be able to:**

- Use the prebuilt, optimised LULESH executable and its matching source information.
- Establish a baseline and collect Hotspots data.
- Interpret sampled CPU time, elapsed time, and runtime waiting.
- Compare one MPI rank using **1, 2, 4, 6, and 12 OpenMP threads**.
- Choose HPC Performance or Memory Access to test a performance hypothesis.

Allow **5–10 minutes to check the session and module**, then **45 minutes for the core lesson**. Hardware-counter collection and an eight-rank example are optional. Keep this notebook beside `lab.py` in the repository's `vtune/` directory.

## Exercise roadmap

| Stop | Time in core lesson | Edit / run | Check before continuing |
|---|---:|---|---|
| [ARE setup](#setup) | Before the lesson | Launch form, module, paths and environment | One running session with 12 cores on one node |
| [1. Establish a baseline](#exercise-1) | 8 min | One-thread warm-up and three repetitions | Successful, repeatable LULESH runs |
| [2. Find a hotspot](#exercise-2) | 10 min | Hotspots collection and reports | A function, its cost, and a saved result |
| [3. Interpret the evidence](#exercise-3) | 8 min | Timing example, reports, source | Explain the metric and its denominator |
| [4. Change one variable](#exercise-4) | 12 min | 1, 2, 4, 6, then 12 threads | Repeated timings and a supported conclusion |
| [5. Choose the next analysis](#exercise-5) | 7 min | Hypothesis worksheet; optional counters | A discriminating next measurement |
| [Extension: eight MPI ranks](#mpi-extension) | Optional | Eight ranks, one thread each | Eight application ranks within the same session |

Run the cells **in order and one experiment at a time**. The kernel launches the commands in the existing ARE job. Keep other notebooks and terminals idle during measurements: Jupyter, VTune, and LULESH share the session's resources. There are no precomputed performance results; fill in the worksheet from your own runs.

<a id="foundations"></a>
## 1. What does a profiler tell us?

A **timer** measures a defined interval. A **profiler** attributes cost to functions, source locations, threads, or hardware events. A hotspot is a place where substantial measured cost accumulates; it is a starting point for investigation.

### Sampling: take snapshots

For example, a profiler might check every **10 milliseconds** and record the executing function, instruction, and **call stack**. If **600 of 1,000 samples** land inside `calculate_forces()`, that suggests roughly **60% of the sampled CPU time** was spent there. This is a statistical estimate, not a count of function calls. Short runs can have too few samples to support a stable ranking.

### Instrumentation: record selected events

Instrumentation adds measurements at points such as function entry and exit. If **100 sequential calls** to `calculate_forces()` accumulate **2 seconds** of elapsed time, the average is **20 milliseconds per call**. Timestamp-based measurements describe elapsed time. Executing the measurement code adds overhead; sampling also has a cost.

```text
Correct result → repeatable baseline → collect → locate expensive code
      ↑                                             |
      └──── validate and re-time ← change one thing ← hypothesis
```

**Predict:** does the most frequently called function necessarily consume the most CPU time?

<details><summary>Check your reasoning</summary>

No. A million calls costing 100 nanoseconds of CPU time each total 0.1 seconds; ten calls costing 100 milliseconds each total 1 second. Measure an unprofiled baseline, then use the profile to investigate where the cost arises.

</details>

<a id="setup"></a>
## 2. Launch ARE with the prebuilt LULESH module

For this workshop the instructor provides an optimised MPI/OpenMP executable as the **private `lulesh` module for `vp91` participants**. Compilation is covered in optional instructor notes; the practical lesson starts with the supplied executable.

1. Sign in to [ARE](https://are.nci.org.au/) and select **JupyterLab**.
2. Select **12 CPU cores on one node**, project **`vp91`**, and the storage listed below.
3. Expand **Advanced options**. In **Modules**, enter the module names below. If the private module directory is not already available, put the directory supplied by the instructor in **Module directories**.
4. Select **Launch**, wait until the session is running, then select **Open JupyterLab**. Open this notebook with a Python 3 kernel.

| ARE setting | Training selection |
|---|---|
| Compute resources | **12 CPU cores, one node** |
| Queue | Instructor's CPU queue (for example, `normal`) |
| Walltime | Enough for the lesson and extensions; **2 hours** is a suggested session length |
| Memory | The instructor's setting / memory associated with the selected compute size |
| Project | `vp91` |
| Storage | `scratch/vp91`, plus any storage containing the private module, executable and source |
| Module directories | Instructor's private module directory, **only if not already discoverable** |
| Modules | `lulesh intel-vtune/2024.2.1` (use the instructor's confirmed VTune version) |

The **Modules** field takes names, without the `module load` prefix. Its terminal equivalent is:

```bash
module load lulesh
module load intel-vtune/2024.2.1
command -v lulesh2.0 mpirun vtune
```

The private module supplies `lulesh2.0` on `PATH` and its compatible Open MPI/compiler runtime. The notebook uses those runtimes. The instructor should also expose the matching source directory through `LULESH_SOURCE`. The final module directory and installation paths are supplied for the session; no particular shared installation path is assumed here.

**Configure modules before ARE launches the kernel.** Loading them later in a separate terminal or `!module load` cell does not update an already running Python kernel. If the module is missing from the kernel, correct the ARE launch settings and relaunch. All profiling commands run inside the existing ARE allocation.

See [NCI's JupyterLab training guide](https://nci900-training-organisation.github.io/learning-resources/Getting-Started/use-jupyterlab.html) for the launch form workflow.

### How the notebook uses the allocation

```text
Your browser → ARE submits one 12-core, single-node job
                   └─ JupyterLab and this Python kernel
                        └─ mpirun → [VTune →] prebuilt LULESH
                             └─ results in your own scratch directory
```

| Experiment | MPI ranks | OpenMP threads per rank | Application core budget |
|---|---:|---:|---:|
| Baseline / first Hotspots | 1 | 1 | 1 |
| Thread comparison | 1 | 2, 4, 6, 12 | 2, 4, 6, 12 |
| Second Hotspots / optional hardware analyses | 1 | 12 | 12 |
| Optional MPI coverage | 8 | 1 | 8 |

The **allocation remains 12 cores throughout**. A one-thread baseline intentionally uses a subset. Twelve allocated cores do not imply twelve MPI ranks: LULESH requires cubic rank counts, so its twelve-core run here uses **one rank × twelve threads**. Eight ranks × two threads would exceed this session's budget.

### Locate the training files and module executable

Use your own notebook copy so you can save answers. If the instructor has not provided a copy of this training repository, obtain it in a JupyterLab terminal:

```bash
cd /scratch/vp91/$USER
git clone https://github.com/NCI900-Training-Organisation/intro-to-profiling.git
```

**RUN the next cell.** It finds `lulesh2.0` on the `PATH` set by `module load lulesh`. It reads `LULESH_SOURCE` if the module exports it. **Only edit `REPO` if the training checkout cannot be found**, or set `LULESH_SOURCE` to the matching source location supplied by the instructor if needed.

Results go to **`/scratch/vp91/<your-user>/vtune-results/`**. The shared executable/source installation is used for reading and execution; it is not a results directory. Keep `ALLOCATED_CORES = 12`.

[lab.py](lab.py) prints each command, checks resources, and gives each run a fresh directory. It uses the module's **Open MPI** launcher, core binding, `OMP_PLACES=cores`, and `OMP_PROC_BIND=close`.

In [ ]:
from pathlib import Path
import getpass
import json
import os
import shlex
import shutil
import socket
import statistics
import subprocess
import sys

PROJECT = "vp91"
WORK = Path(f"/scratch/{PROJECT}/{getpass.getuser()}")
candidates = [Path.cwd(), *Path.cwd().parents, WORK / "intro-to-profiling"]
REPO = next((p for p in candidates if (p / "vtune/lab.py").is_file()), None)
# If needed: REPO = Path("/scratch/vp91/your_user/intro-to-profiling")
if REPO is None:
    raise FileNotFoundError("Set REPO to the training checkout containing vtune/lab.py")
REPO = REPO.expanduser().resolve()

executable = shutil.which("lulesh2.0")
if executable is None:
    raise RuntimeError("lulesh2.0 is missing from the kernel PATH. Add lulesh to the ARE "
                       "Modules field, check its private module directory, and relaunch.")
LULESH_EXE = Path(executable).resolve()
source_setting = os.environ.get("LULESH_SOURCE")
LULESH_SOURCE = Path(source_setting).expanduser().resolve() if source_setting else None
# Optional: LULESH_SOURCE = Path("/matching/source/path/from/instructor")
RESULTS = WORK / "vtune-results"
ALLOCATED_CORES = 12
THREAD_COUNTS = [1, 2, 4, 6, 12]
PROFILE_THREADS = 12
sys.path.insert(0, str(REPO / "vtune"))
from lab import Lab, report, result_dirs

lab = Lab(binary=LULESH_EXE, results=RESULTS, allocated_cores=ALLOCATED_CORES,
          ranks=1, size=30, iterations=200)
print("Training repository:", REPO)
print("Module executable:", LULESH_EXE)
print("Matching source:", LULESH_SOURCE or "not supplied by the module")
print("Run budget:", ALLOCATED_CORES, "cores on one node")
print("Your results:", RESULTS)

In [ ]:
# Inspect this running session; this cell does not submit a job.
print("Host:", socket.gethostname())
print("PBS job:", os.environ.get("PBS_JOBID", "not exposed to the kernel"))
print("PBS_NCPUS:", os.environ.get("PBS_NCPUS", "not exposed to the kernel"))

pbs_cores = os.environ.get("PBS_NCPUS")
if pbs_cores is not None:
    if not pbs_cores.isdigit() or int(pbs_cores) != ALLOCATED_CORES:
        raise RuntimeError("This lesson expects the ARE session launched with 12 CPU cores")
else:
    print("Use the ARE session card to verify 12 cores; the lab budget remains 12.")

nodefile_name = os.environ.get("PBS_NODEFILE")
if nodefile_name:
    nodefile = Path(nodefile_name)
    if nodefile.is_file():
        hosts = {line.strip() for line in nodefile.read_text().splitlines() if line.strip()}
        print("Hosts in the allocation:", sorted(hosts))
        if len(hosts) != 1:
            raise RuntimeError("This workshop uses one node; check the ARE session settings")
    else:
        print("PBS_NODEFILE is not readable here; check the ARE session card for one node.")
else:
    print("Node list not exposed; check the ARE session card for one node.")

if hasattr(os, "sched_getaffinity"):
    visible_cpus = len(os.sched_getaffinity(0))
    print("Logical CPUs in this kernel's affinity mask:", visible_cpus)
    if visible_cpus < ALLOCATED_CORES:
        raise RuntimeError("Kernel affinity is too narrow for 12 threads; ask the instructor to check the session")
print("All experiments are capped at 12 cores, even if the host exposes more CPUs.")

In [ ]:
# Check runtime tools inherited from the ARE Modules setting.
required = ("lulesh2.0", "mpirun", "vtune")
missing = [tool for tool in required if not shutil.which(tool)]
if missing:
    raise RuntimeError("Missing tools: " + ", ".join(missing) +
                       ". Check lulesh/VTune and their runtime dependencies in ARE.")
mpi_version = subprocess.check_output(["mpirun", "--version"], text=True)
print(mpi_version)
if "Open MPI" not in mpi_version and "OpenRTE" not in mpi_version:
    raise RuntimeError("The private lulesh module must supply the matching Open MPI runtime")
vtune_version = subprocess.check_output(["vtune", "-version"], text=True)
print(vtune_version)
lab.validate(threads=PROFILE_THREADS)
if LULESH_SOURCE is None or not LULESH_SOURCE.is_dir():
    print("Matching source is unavailable. Profiling can proceed; obtain the source path "
          "from the instructor for Exercise 3 and set LULESH_SOURCE.")
lab.results.mkdir(parents=True, exist_ok=True)
(lab.results / "environment.txt").write_text(
    mpi_version + "\n" + vtune_version +
    "\nModules: " + os.environ.get("LOADEDMODULES", "unknown") +
    f"\nExecutable: {LULESH_EXE}\nSource: {LULESH_SOURCE}\n" +
    f"ARE allocation: {ALLOCATED_CORES} cores on one node\n")

### Why the supplied build is suitable for profiling

The instructor prepares the `lulesh` module with **MPI and OpenMP enabled** and an **optimised executable with debug symbols**. Participants use the same supplied binary for all comparisons.

| Build setting | Why it matters when reading a profile |
|---|---|
| `-O3` | Represents optimised application execution |
| `-g` | Retains symbols and source locations |
| `-fno-omit-frame-pointer` | Helps stack unwinding |
| MPI and OpenMP enabled | Supports the rank and thread experiments |

The module's source directory must match the executable. Optimisation/inlining can make source-line attribution less literal. No `-pg` instrumentation is needed for VTune.

**There is no build step in the participant workflow.** Optional preparation instructions for the module maintainer are in the [application setup notes](../docs/source/tutorial/intro.rst).

### LULESH: what is the workload?

LULESH is a C++ hydrodynamics proxy application. MPI partitions its three-dimensional mesh; OpenMP threads work within each rank.

- `-s N` sets the mesh side length **per rank**: total elements are `ranks × N³`.
- `-i K` limits the number of iterations; the simulation may reach its stop time sooner. Record the **actual** iteration count printed at completion.
- The MPI decomposition requires **1, 8, 27, … ranks**, a perfect cube. Four ranks are invalid for this application.

Start with **one rank** and vary threads. That holds the mesh and decomposition fixed and makes an introductory comparison easier to interpret. For multi-rank runs, use the launcher **outside** VTune so each application rank is profiled. With Open MPI, `-trace-mpi` enables MPI-aware result naming; results may acquire a hostname suffix. [LLNL source](https://github.com/LLNL/LULESH/blob/master/lulesh-init.cc#L636-L651), [Intel MPI profiling guide](https://www.intel.com/content/www/us/en/docs/vtune-profiler/user-guide/2023-2/mpi-code-analysis.html).

**Setup checkpoint:** the ARE session has **12 cores on one node**; `lulesh2.0`, `mpirun`, and `vtune` are visible to the kernel; the twelve-thread resource check passes; and your personal results directory is writable. Record the source path if it is available.

`os.cpu_count()` or `lscpu` can describe more of the node than your allocation. The lab uses the **12-core budget selected in ARE**. Run the one-thread warm-up next to check that the supplied executable and runtime work together. Resolve missing-module, library or launch errors with the instructor before profiling.

<a id="exercise-1"></a>
## Exercise 1 — Establish a baseline (8 min)

**RUN the next cell** as a warm-up and size check. Inspect the complete `run.log` at the printed path. Find the rank count, thread count, actual iterations, final energy/symmetry diagnostics, and LULESH's elapsed time.

Start with `size=30`, `iterations=200`. Aim for several seconds of **application execution**, preferably about 5–20 seconds for this lesson. If it is too short, increase `lab.size` and/or `lab.iterations` in the configuration cell, then rerun configuration and warm-up. For example, try `size=45`, `iterations=1000`; reaching the physical stop time can limit the benefit of raising only the iteration cap. Keep the final choice fixed for all comparisons.

The helper records **launcher wall time** around `mpirun`. This includes startup and shutdown. LULESH's own timer covers a different region. Label the timer you use and compare the same region in every run.

In [ ]:
warmup = lab.run(threads=1, label="warmup")
print(f"Launcher wall time: {warmup['launcher_wall_seconds']:.3f} s")

Once the workload is suitable, **run three unprofiled repetitions**. The runs use fresh directories, so rerunning a cell does not overwrite a profile. Logs include both the application output and binding messages; `run.json` records the command, module list, OpenMP settings, executable hash, and timing.

In [ ]:
baselines = [lab.run(threads=1) for _ in range(3)]
times = [r["launcher_wall_seconds"] for r in baselines]
print("Launcher wall times (s):", [round(t, 3) for t in times])
print(f"Median {statistics.median(times):.3f} s; range {min(times):.3f}–{max(times):.3f} s")

**Exercise 1 checkpoint:** all runs finish successfully; the actual iteration count and numerical diagnostics agree within an appropriate floating-point tolerance for this fixed workload. A zero exit code alone is not a numerical validation. Explain any substantial timing spread before proceeding.

**Record:** mesh size, iteration cap and actual iterations, binary/build, rank/thread counts, binding, timing region, three timings, and a log path in the [worksheet](#worksheet).

<a id="hotspots"></a>
## 3. Hotspots: locate expensive code

Start with software sampling (`sampling-mode=sw`). It collects application samples and call stacks without hardware sampling drivers. It still requires the platform's normal process-profiling permissions and adds overhead. Hardware sampling is a separate mode and needs supported counters and access permissions. [Intel Hotspots documentation](https://cdrdv2-public.intel.com/824400/vtune-profiler_user-guide_2024.2-766319-824400.pdf).

The helper also sets `enable-characterization-insights=false`. VTune enables additional performance insights by default; these collect hardware events even when software sampling is selected. Disabling them avoids that extra hardware requirement and associated PMU setup failures.

The command structure used by the next exercise is:

```bash
export OMP_NUM_THREADS=1 OMP_DYNAMIC=FALSE
export OMP_PLACES=cores OMP_PROC_BIND=close
mpirun -np 1 --map-by slot:PE=1 --bind-to core \
  vtune -collect hotspots -knob sampling-mode=sw \
  -knob enable-characterization-insights=false \
  -trace-mpi -result-dir /path/to/fresh/result \
  -- lulesh2.0 -s 30 -i 200
```

`--` separates collector options from the application. The notebook substitutes your configured workload and uses the module executable and creates the fresh result parent automatically. It also adds `--report-bindings` to `mpirun`.

<a id="exercise-2"></a>
## Exercise 2 — Find a hotspot (10 min)

1. Predict whether application calculations or runtime functions will dominate at one thread.
2. **RUN** the collection, summary, and function report cells below in order.
3. Identify the three most expensive functions and their CPU times. Separate LULESH code from MPI/OpenMP/runtime code.
4. Choose one function to follow to its callers and source in Exercise 3.

A **summary** gives the overall run metrics; a **hotspots report grouped by function** gives the function costs. Inspect any warnings in `run.log`, particularly incomplete collection or missing symbols.

In [ ]:
hotspots_1 = lab.run(threads=1, analysis="hotspots")
print("Actual result directories:")
for result in result_dirs(hotspots_1):
    print(result)

In [ ]:
report(hotspots_1, "summary")

In [ ]:
report(hotspots_1, "hotspots")

**Exercise 2 checkpoint:** the report contains your LULESH process and named functions, and you can give the saved result path. Use the exact directory printed above, including any hostname suffix. Do not infer a hotspot ranking from this notebook: it depends on the build, input, and machine.

The helper's time for a profiled run includes collection and finalisation. It is **not** a clean measure of application slowdown due to sampling; use comparable application timing regions for that question.

<a id="metrics"></a>
## 4. Read metrics before drawing conclusions

| Metric | Interpretation | Avoid this inference |
|---|---|---|
| Elapsed time | Wall time for the reported collection interval | It must equal the application's own timer |
| CPU time | On-CPU time accumulated across threads | It cannot exceed elapsed time |
| Effective time | CPU time excluding recognised spin/runtime overhead | Every instruction is productive, or memory stalls are absent |
| Spin time | Recognised active waiting on the CPU | All MPI time is communication cost |
| Overhead time | Recognised parallel-runtime management | This measures all profiler overhead |
| Total thread count | Threads observed, including possible helper threads | All those threads ran simultaneously |
| Paused time | Intervals where collection was paused | It is a general measure of sleeping/descheduled threads |

Not all metrics are available for every analysis, CPU, or threading runtime. GCC/OpenMP builds may lack the detailed region/efficiency metrics available with Intel's instrumented runtime. Treat an unavailable metric as unavailable, not zero. [Intel CPU metric definitions](https://www.intel.com/content/www/us/en/docs/vtune-profiler/user-guide/2023-1/cpu-metrics-reference.html), [collection pauses](https://www.intel.com/content/www/us/en/docs/vtune-profiler/user-guide/2024-0/problem-unexpected-paused-time.html).

### A denominator changes the story

**Illustrative arithmetic, not a captured VTune result:** four threads each spend 6 seconds on the CPU during an 8-second interval. Total CPU time is 24 seconds; average on-CPU concurrency is `24 / 8 = 3`. That includes spin and hides changes over time.

Now suppose a report shows **6 effectively used physical cores out of 48 detected cores**. That is `6 / 48 = 12.5%` relative to the node. Your ARE session has **12 allocated cores**, so the same average corresponds to `6 / 12 = 50%` of that allocation, provided the quantities use the same physical-core basis. Neither percentage is itself measured parallel efficiency. Check the allocation and metric definition before interpreting a low node-wide percentage.

**Fixed-work parallel efficiency:** with one MPI rank and `p` OpenMP threads, `S_p = T_1 / T_p` and `E_p = S_p / p`, using comparable unprofiled elapsed times. For the twelve-thread experiment, use `p=12`; for the four-thread experiment, use `p=4`, even though the allocation remains twelve cores throughout.

<a id="exercise-3"></a>
## Exercise 3 — Interpret and inspect (8 min)

**ANSWER first:** for the illustrative 24 CPU-seconds / 8 elapsed-seconds example:

1. What is the average number of on-CPU threads?
2. Can you conclude that all four threads did useful work throughout?
3. Would zero paused time prove that the program never slept?

<details><summary>Reference reasoning — open after attempting</summary>

1. Three on average over the same interval.
2. No: CPU time may include spin, and this average hides changes over time.
3. No: paused collection and sleeping threads are different concepts.

</details>

**Use your saved text reports for the ARE exercise.** The next cell finds the C++ source files containing the function name you select. Open a matching file in the JupyterLab text editor and inspect the surrounding loop and calls. If the shared directory is outside the Jupyter file browser, inspect it from a terminal or copy the file to your own directory for reading; preserve the shared source. Record the function's CPU time from the report and a question about the code.

A normal ARE **JupyterLab** session does not by itself provide a desktop display for `vtune-gui`. GUI exploration is an optional instructor demonstration or later activity using a configured desktop/VNC session or compatible local VTune installation. The core exercise is completed in this JupyterLab session using reports and source.

In [ ]:
FUNCTION_NAME = "CalcForceForNodes"  # EDIT: use a function from your Hotspots report
if LULESH_SOURCE is None or not LULESH_SOURCE.is_dir():
    print("Source inspection is unavailable: ask the instructor for the matching source "
          "directory and set LULESH_SOURCE. Continue interpreting the saved function report.")
else:
    matches = []
    for source in sorted(LULESH_SOURCE.glob("*.cc")):
        for line_number, line in enumerate(source.read_text(errors="replace").splitlines(), 1):
            if FUNCTION_NAME in line:
                matches.append((source, line_number, line.strip()))
    for source, line_number, line in matches[:20]:
        print(f"{source}:{line_number}: {line}")
    if not matches:
        print("No match: use the base C++ function name, without arguments or an OpenMP suffix.")

print("\nOptional GUI commands for a display-enabled environment:")
for result in result_dirs(hotspots_1):
    print(shlex.join(["vtune-gui", str(result)]))

### Checkpoint and optional GUI route

**Exercise 3 checkpoint:** record a function, its CPU time, its source location, and one explanation to test. Reading the source alongside a function report does not establish which exact source line received samples.

When a suitable display is available, open the result and follow:

1. **Summary:** target, elapsed time, and warnings.
2. **Bottom-up:** expensive function, with process/thread grouping as needed.
3. **Caller/Callee:** how the function is reached; own cost versus cost including callees.
4. **Source/Assembly:** sampled operations; add the matching source path if necessary.
5. **Timeline:** phases and thread behaviour hidden by an average.

Record whether your observations came from text reports/source or from the GUI. Copy the complete result and matching executable/source together for later analysis.

<a id="exercise-4"></a>
## Exercise 4 — Use the twelve-core allocation (12 min)

**Predict:** how will runtime change at **1, 2, 4, 6, and 12 threads**? Why could CPU time rise while elapsed time falls?

Keep the executable, mesh, iteration cap, **one MPI rank**, and binding policy fixed. The allocation stays at twelve cores; only the application's thread count changes. The next cell warms up each new configuration and measures three unprofiled runs. It reuses the one-thread baseline from Exercise 1. Runs are sequential, so their core requests do not add together.

Check the printed LULESH thread count, numerical diagnostics and actual iterations. If you changed the workload after Exercise 1, rerun the baseline first.

In [ ]:
expected = (lab.ranks, lab.size, lab.iterations)
if any((r["ranks"], r["size"], r["iterations"]) != expected for r in baselines):
    raise ValueError("Workload changed: rerun Exercise 1 before comparing")
for threads in THREAD_COUNTS:
    lab.validate(threads)
measurements = {1: baselines}
for threads in THREAD_COUNTS[1:]:
    lab.run(threads=threads, label="warmup")
    measurements[threads] = [lab.run(threads=threads) for _ in range(3)]

print("threads   median launcher s   min s    max s    speedup   efficiency")
t1 = statistics.median(r["launcher_wall_seconds"] for r in measurements[1])
for threads, runs in measurements.items():
    timings = [r["launcher_wall_seconds"] for r in runs]
    median = statistics.median(timings)
    speedup = t1 / median
    print(f"{threads:7d} {median:19.3f} {min(timings):7.3f} {max(timings):8.3f} "
          f"{speedup:10.2f} {speedup / threads:11.1%}")

These speedups include MPI launch overhead. Also compare application times in `run.log` if launch overhead is material. If timing variation is comparable to the apparent improvement, repeat or increase the workload before concluding.

**RUN a second Hotspots collection at twelve threads**, then compare it with the one-thread result. Jupyter and the collector share the session: keep other work idle. Compare absolute function times as well as percentages; a function's percentage can rise because another function got faster.

In [ ]:
hotspots_12 = lab.run(threads=PROFILE_THREADS, analysis="hotspots")
report(hotspots_12, "summary")
report(hotspots_12, "hotspots")

**Exercise 4 checkpoint:** explain whether the repeated unprofiled timings support your prediction. Recheck the numerical diagnostics before claiming an improvement. Use the profiles to suggest a reason; the change in timing alone does not identify the cause.

<details><summary>Possible interpretations — there is no prescribed winner</summary>

A small mesh may have insufficient work per parallel region. Synchronisation, imbalance, and thread management may outweigh the benefit of additional threads. A larger workload can behave differently. Rising CPU time can reflect additional parallel work or waiting; inspect functions and the timeline. A flat speedup curve alone does not establish a memory-bandwidth bottleneck.

</details>

<a id="next-analysis"></a>
## 5. Choose an analysis to test the explanation

| Question | Collection | Evidence to seek |
|---|---|---|
| Where is CPU time spent? | `hotspots` | Functions, callers, thread activity |
| How are compute resources used? | `hpc-performance` | Available CPU, memory, floating-point and parallel metrics |
| Is access to data limiting a hot region? | `memory-access` | Available memory-stall, cache, bandwidth and NUMA metrics |

HPC Performance and Memory Access need supported hardware events and suitable access to the counters. The available metrics depend on the processor, VTune version, and runtime. Hardware data may be unavailable in a particular compute or ARE environment. Read the collection error rather than changing system settings.

For memory behaviour, distinguish **bandwidth** (data transferred per second) from **latency** (time waiting for an access). A memory-stalled region can have low bandwidth if it issues too few independent requests. High traffic alone does not prove that reducing traffic will improve elapsed time. [Intel HPC collection guide](https://www.intel.com/content/www/us/en/docs/vtune-profiler/user-guide/2023-1/run-hpc-performance-analysis-command-line.html), [Intel Memory Usage guide](https://www.intel.com/content/www/us/en/docs/vtune-profiler/user-guide/2025-1/memory-usage-view.html).

<a id="exercise-5"></a>
## Exercise 5 — State a testable hypothesis (7 min)

**Complete this task even if hardware counters are unavailable.** Use your observed hotspot and thread comparison to fill in one row:

| Observation | Hypothesis | Next analysis / comparison | Evidence that would weaken the hypothesis |
|---|---|---|---|
| Your measurement | Your explanation | One targeted test | An alternative outcome |

Example reasoning, **not a LULESH measurement**: “More threads did not reduce runtime; perhaps a barrier is waiting for one late thread. Compare per-thread work and waiting in the timeline. Similar progress across threads with little waiting would weaken that explanation.”

**Optional collection:** after the instructor confirms counter access, set `RUN_HARDWARE = True` below. The two analyses run separately on the same workload. If a collection fails, preserve its log, return the flag to `False`, and continue to the worksheet. Do not treat the missing metrics as zero or fabricate a substitute report.

In [ ]:
RUN_HARDWARE = False  # EDIT only when hardware collection is available in this session
if RUN_HARDWARE:
    hpc = lab.run(threads=PROFILE_THREADS, analysis="hpc-performance")
    report(hpc, "summary")
else:
    print("Optional HPC Performance collection skipped; complete the hypothesis worksheet.")

In [ ]:
if RUN_HARDWARE:
    memory = lab.run(threads=PROFILE_THREADS, analysis="memory-access")
    report(memory, "summary")
else:
    print("Optional Memory Access collection skipped.")

**Exercise 5 checkpoint:** identify one available metric that bears on your hypothesis, including its unit and denominator. If hardware data is unavailable, record that limitation and the next measurement you would request. End with an action that could be tested by another unprofiled timing and correctness check.

<a id="mpi-extension"></a>
## Optional extension — Eight ranks inside the same ARE session

Use **eight MPI ranks × one OpenMP thread**, consuming up to eight of the twelve allocated cores. This fits the existing single-node session. **Twelve ranks are invalid for LULESH**, and eight ranks × two threads would request sixteen cores, exceeding the budget.

The helper starts VTune for the application ranks and passes `-trace-mpi`. Inspect all returned result directories and confirm eight application ranks. Leave `ALLOCATED_CORES=12`; set `RUN_MPI_EXTENSION=True` to run this extension after the other experiments finish.

This demonstrates **MPI process coverage**. Holding `-s` fixed gives eight times as many elements as the one-rank run, so these timings do not form a fixed-work scaling comparison. For the same global element count, one rank with `-s 60` and eight ranks with `-s 30` would need further checks of actual iterations and comparable physics/work.

In [ ]:
RUN_MPI_EXTENSION = False
if RUN_MPI_EXTENSION:
    mpi_lab = Lab(binary=lab.binary, results=lab.results,
                  allocated_cores=ALLOCATED_CORES, ranks=8,
                  size=lab.size, iterations=lab.iterations)
    mpi_lab.run(threads=1, label="mpi-baseline")
    mpi_hotspots = mpi_lab.run(threads=1, analysis="hotspots")
    report(mpi_hotspots, "summary")
    for result in result_dirs(mpi_hotspots):
        print(shlex.join(["vtune-gui", str(result)]))
else:
    print("Optional eight-rank collection skipped.")

**Extension checkpoint:** confirm eight application ranks in the log/result, locate the application and MPI routines, and explain why a large MPI cost could include waiting for work on another rank. Do not equate all MPI time with network transfer time.

<a id="worksheet"></a>
## Record your measurements

**EDIT this Markdown cell** after each exercise. Mark missing metrics `not collected` or `not available`.

ARE session: **12 CPU cores, one node**. Job ID / hostname: …

Loaded `lulesh` module / executable path / binary hash: …

Matching source / build details supplied by the instructor: …  MPI / VTune versions: …

Mesh side per rank: …  Iteration cap / actual iterations: …  Numerical checks / tolerance: …

| Run | Ranks × threads | Timer / 3 unprofiled times / median | Dominant function and CPU time | Log / result path |
|---|---|---|---|---|
| Baseline | 1 × 1 | | | |
| Thread comparison | 1 × 2 | | | |
| Thread comparison | 1 × 4 | | | |
| Thread comparison | 1 × 6 | | | |
| Thread comparison | 1 × 12 | | | |
| Hotspots | 1 × 1 | Profile; separate from baseline | | |
| Hotspots | 1 × 12 | Profile; separate from baseline | | |
| HPC / memory (optional) | 1 × 12 | | | |
| MPI coverage (optional) | 8 × 1 | Different total mesh | | |

**Does using all twelve cores improve this workload? Evidence and variability:** …

**My hypothesis and the next change I would test:** …

**Success criterion (correctness and elapsed time):** …

## Troubleshooting

| Symptom | Next check |
|---|---|
| Session reports a different core count | Check the ARE session card; relaunch with 12 cores on one node |
| Storage or repository path is missing | Check Storage in the ARE launch form and your own project/user directory |
| `lulesh2.0` missing | Check `lulesh` in ARE Modules, the private Module directories setting, and mounted storage; relaunch |
| LULESH reports a missing shared library | Ask the instructor to check the private module runtime dependencies |
| `LULESH_SOURCE` unset or unreadable | Obtain the matching source path; function-level profiling can continue |
| `vtune` missing inside the notebook | Correct Modules in the ARE launch form and relaunch the session |
| MPI binding error or wrong thread count | Open MPI version, allocated cores, `--map-by`, `OMP_NUM_THREADS` |
| “Num processors must be a cube” | Use 1 or 8 ranks; the 12-core run is 1 rank × 12 threads |
| No result at the requested name | Check the actual hostname-suffixed result path and collection log |
| Hotspots fails with an internal `pmu-type` error | Check that `lab.py` adds `-knob enable-characterization-insights=false`; after updating the helper, restart the kernel and rerun setup |
| Hardware events denied/unavailable | Retain the error; continue with software Hotspots and discuss with the instructor |
| Unknown functions or no source | Ask the instructor to verify the prebuilt binary symbols and matching source path; inspect finalisation warnings |
| Few samples / unstable rankings | Increase application work, then rerun baseline and profile at the same size |
| Low whole-node utilisation | Compare detected node cores with this session's 12-core budget |
| GUI cannot open | Use reports/source in JupyterLab; GUI needs a separate display-enabled environment |

## Final concept checks

1. Why use optimisation and debug symbols together?
2. How can CPU time exceed elapsed time?
3. What remains unknown after finding the most expensive function?
4. Why do the helper's profiled launcher time and VTune's elapsed time differ?
5. Why does keeping `-s` fixed while increasing ranks change the total work?
6. What evidence would persuade you that your next change improved the application?
7. Why can this 12-core session run 1 rank × 12 threads, but not 12 LULESH ranks?

<details><summary>Reference answers</summary>

1. Preserve realistic execution while retaining source attribution.
2. CPU time sums activity across threads.
3. Its cause, scope for improvement, and effect on end-to-end runtime.
4. The launcher measurement includes startup, collection, and result finalisation; the profiler reports its collection interval.
5. `-s` describes a local cube, so total elements are `ranks × s³`.
6. Correct numerical behaviour and a repeatable improvement for the same workload and timing region, with the environment and changes recorded.
7. The resource budget is ranks × threads, while LULESH separately requires a cubic MPI rank count. One rank is valid; twelve ranks are not.

</details>

## Sources and teaching provenance

The teaching sequence comes from the repository's [application setup](../docs/source/tutorial/intro.rst) and [Intel VTune tutorial](../docs/source/tutorial/vtune.rst), with concept/exercise/checkpoint structure modelled on [MPI Day 1](../mpi/day1/mpi_lab1.ipynb) and [MPI Day 2](../mpi/day2/mpi_lab2.ipynb). The slide styling and NCI logo come from `NCI_MPI_2026(1).zip`.

ARE launch settings follow the [NCI JupyterLab training guide](https://nci900-training-organisation.github.io/learning-resources/Getting-Started/use-jupyterlab.html) and the supplied ARE form example. The **12-core, single-node allocation** and **private `lulesh` module for `vp91`** are the workshop configuration. The module should expose `lulesh2.0` and the matching runtime on `PATH`, and preferably `LULESH_SOURCE` for source inspection.

Technical checks use primary documentation:

- [LLNL LULESH source and build instructions](https://github.com/LLNL/LULESH).
- [Intel VTune 2024.2 User Guide](https://cdrdv2-public.intel.com/824400/vtune-profiler_user-guide_2024.2-766319-824400.pdf): Hotspots and analysis modes.
- [Intel MPI profiling cookbook](https://www.intel.com/content/www/us/en/docs/vtune-profiler/cookbook/2024-0/profiling-mpi-applications.html): launch order, MPI result directories and GUI workflow.
- [Intel CPU metrics](https://www.intel.com/content/www/us/en/docs/vtune-profiler/user-guide/2023-1/cpu-metrics-reference.html) and [paused collection](https://www.intel.com/content/www/us/en/docs/vtune-profiler/user-guide/2024-0/problem-unexpected-paused-time.html).

Corrections to the original RST examples: use cubic LULESH rank counts; launch VTune per rank; interpret paused time as a collection state; qualify effective-time and utilisation metrics. Module versions remain workshop examples. No Gadi measurements are bundled with this notebook.